# Results_paper_stats

Builds the image-level market activity file and makes four paper figures.

| Paper exhibit | Cell | File (in `output_paper/paper_exhibits`) |
|---|---|---|
| Figure 3, panel a: spatial distribution of periodic markets | `4abba201` | `fig03a_market_map.png` |
| Figure 3, panel b: day of market activity | `67b2d0b0` | `fig03b_days_of_operation.png` |
| Figure 7, panel a: market activity and real GDP | `079d7ef2` | `fig07a_mai_gdp.png` |
| Figure 7, panel b: market activity and nighttime lights | `ec3f930c` | `fig07b_mai_ntl.png` |

Cell `e568e199` also writes `processed_data/market_images.csv`, which the other
notebooks read. EXHIBITS.md at the repository root lists every exhibit in the paper.
Cells for figures not in the final paper are in `archive/Results_paper_stats_archive.ipynb`.

In [ ]:
import os
import sys
import glob
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from scipy import stats as _st

# Locate code_paper/NTL_processing/config.py whether the kernel was started in
# code_paper itself or one level above it.
_candidates = [os.getcwd(), os.path.join(os.getcwd(), "code_paper"),
               os.path.dirname(os.getcwd())]
NB_DIR = next((p for p in _candidates
               if os.path.exists(os.path.join(p, "NTL_processing", "config.py"))), None)
if NB_DIR is None:
    raise RuntimeError("Could not find NTL_processing/config.py. Start the kernel in "
                       "the code_paper folder, or set MAI_BASE.")
sys.path.insert(0, os.path.join(NB_DIR, "NTL_processing"))
sys.path.insert(0, os.path.join(NB_DIR, "exhibits"))

# Paper figures are drawn by exhibits/plots.py in the house style of exhibits/style.py;
# each figure cell below computes its data, saves it (S.save_data) and calls P.<figure>.
import style as S
import plots as P


from config import (BASE, DATA, PROCESSED, OUTPUT, PAPER_EXHIBITS, VIIRS_DIR, MASK_DIR,
                    KABUL_LON, KABUL_LAT, BUFFER_KM, MAX_RAD_CAP)

# Paper figures go to the shared paper_exhibits folder, named by exhibit number.
OUT_PAPER = PAPER_EXHIBITS

# Province boundaries (ADM1), carrying the CSO region each province belongs to.
REPO_DATA = os.path.join(BASE, "document", "Repo_09042026", "data")
ADM1 = os.path.join(REPO_DATA, "ADM1_Prov_2017_NSIA", "ADM1_Prov_2017_NSIA.shp")

REGIONS = ["Central", "East", "NorthEast", "NorthWest", "SouthEast", "SouthWest", "West"]
RNAME = {"NorthEast": "North East", "NorthWest": "North West",
         "SouthEast": "South East", "SouthWest": "South West"}

# National real GDP series (Sector x year), used by Figure 7a.
MACRO = os.path.join(DATA, "macro_series.xlsx")

# Province monthly nightlights (civilian + coalition), used by Figure 7b; only
# shipped inside the replication repo, not the main data tree.
NTL_ADU = os.path.join(REPO_DATA, "raw_nightlights_ADU", "NTL_ADU2.xlsx")

YEARS = list(range(2018, 2026))

print("base      :", BASE)
print("processed :", PROCESSED)
print("paper exhibits:", OUT_PAPER)

## 1. Load market data

Loads the different market data to create a single file

In [18]:
# Build the image-level market activity dataset for all 137 detected markets.
# One row per market per satellite image, market days AND non-market days.
#
# Sources (both raw):
#   ACT_ALL  forAFG_paper_activity_all.csv -- image-level activity file, the only
#            source carrying non-market days.
#   SHP      afg_periodic_markets_20250827.shp -- detected market polygons, used
#            for market area.
#
# Trading days come from `weekdayThisAreaIsActive`, not the registry; it agrees
# with the registry for all 133 markets where both exist and supplies a day for
# the three the registry leaves blank, so all 137 markets get a schedule. One
# market (Hirat) is overridden to Monday; `mktDay` is recomputed from the final
# day set so the flag and schedule cannot drift apart.
import geopandas as gpd


ACT_ALL = os.path.join(DATA, "afg_periodic_markets_20260213", "forAFG_paper_activity_all.csv")
SHP = os.path.join(DATA, "afg_periodic_markets_20250827", "afg_periodic_markets_20250827.shp")
EQUAL_AREA = "EPSG:32641"                       # UTM 41N, metres, covers Afghanistan
OVERRIDE_DAYS = {"lon62_5018lat34_2416": {1}}   # Hirat: authors supplied Monday only
DOW = {0: "Sunday", 1: "Monday", 2: "Tuesday", 3: "Wednesday",
       4: "Thursday", 5: "Friday", 6: "Saturday"}

process = 1

if process == 1:
    
    def trading_days(a):
        """One set of trading weekdays per market (0 = Sunday), for all 137."""
        d = (a.dropna(subset=["weekdayThisAreaIsActive"])
            .groupby("mktID").weekdayThisAreaIsActive
            .apply(lambda s: frozenset(int(v) for v in s.unique())))
        d = d.reindex(a.mktID.unique())
        for m, days in OVERRIDE_DAYS.items():
            if m in d.index:
                d.loc[m] = frozenset(days)
        assert d.apply(lambda s: isinstance(s, frozenset) and len(s) > 0).all(), \
            f"no trading day for {list(d.index[d.isna()])}"
        return d


    def market_area():
        """Detected-market polygon area, in square metres and hectares."""
        g = gpd.read_file(SHP)[["mktID", "geometry"]]
        g = g[~g.mktID.duplicated()]
        g["market_area_m2"] = g.to_crs(EQUAL_AREA).area
        g["market_area_ha"] = g.market_area_m2 / 10_000
        return g.drop(columns="geometry").set_index("mktID")

    def build_market_images():
        a = pd.read_csv(ACT_ALL, low_memory=False).rename(columns={"Location": "mktID"})
        a["date"] = pd.to_datetime(a.date, format="mixed", errors="coerce")
        a = a.dropna(subset=["date"])

        # ADJUSTMET FOR MISSING INFO MARKETS
        # lon64_1471lat30_8694 lon68_8661lat33_6955 lon65_61lat32_3537
        MISSING_INFO_MARKETS21 = ['lon62_5018lat34_2416', 
                                 'lon64_7616lat31_9764', 'lon64_9931lat32_7535']
        display(a['activity_area_norm2020'].describe())
        a['activity_area_norm2020'] = np.where(a['mktID'].isin(MISSING_INFO_MARKETS21), a['activity_area_norm2021'], a['activity_area_norm2020'])


        display(a['activity_area_norm2020'].describe())
        
        days = trading_days(a)
        a["marketDays"] = a.mktID.map(days.apply(lambda s: "{" + ", ".join(str(d) for d in sorted(s)) + "}"))
        a["n_market_days"] = a.mktID.map(days.apply(len))
        a["market_day_names"] = a.mktID.map(days.apply(lambda s: ", ".join(DOW[d] for d in sorted(s))))

        # calendar weekday on the panel's own coding: 0 = Sunday ... 6 = Saturday
        a["weekday_calc"] = (a.date.dt.dayofweek + 1) % 7
        bad = (a.weekday_calc != a.weekday).mean()
        assert bad < 0.001, f"{bad:.3%} of rows disagree with the file's own weekday"

        a["mktDay_raw"] = a.mktDay                         # as the source file has it
        a["mktDay"] = [int(w in days[m]) for m, w in zip(a.mktID, a.weekday_calc)]
        a["day_type"] = np.where(a.mktDay == 1, "market day", "non-market day")

        a["year"] = a.date.dt.year
        a["month"] = a.date.dt.month
        a["week"] = a.date.dt.isocalendar().week.astype(int)
        a["quarter"] = (a.month - 1) // 3 + 1

        a = a.join(market_area(), on="mktID")


        KEEP = ["mktID", "marketlat", "marketlon", "admLvl1", "un_region",
                "market_area_m2", 
                "marketDays", "n_market_days", "market_day_names",
                "image_id", "instrument", "date", "year", "month", "week", "quarter",
                "weekday", "mktDay", "mktDay_raw", "day_type",
                "activity_area_norm2020","activity_area_norm2021",
                "activity_measure", "activity_measure_area"]
        D = a[[c for c in KEEP if c in a.columns]].copy()
        return D.sort_values(["mktID", "date"]).reset_index(drop=True)

    #, "activity_area_norm2018","activity_area_norm2021", "activity_area_norm2022", "clear_percent", "diff_to_median_time","market_area_ha",

    def validate_market_images(D):
        """Assertions that must hold for the dataset to be usable."""
        ok = []
        ok.append(("137 markets", D.mktID.nunique() == 137))
        mdays = D.groupby("mktID").marketDays.first()
        ok.append(("every market has a trading day", mdays.notna().all()))
        viol = D[D.mktDay == 1].apply(
            lambda r: str(int(r.weekday)) not in r.marketDays.strip("{}").split(", "), axis=1).sum()
        ok.append(("every market-day image falls on a declared trading day", viol == 0))
        ok.append(("one area per market", D.groupby("mktID").market_area_m2.nunique(dropna=False).max() == 1))
        ok.append(("one coordinate pair per market",
                D.groupby("mktID")[["marketlat", "marketlon"]].nunique().max().max() == 1))
        ok.append(("market days score far above non-market days",
                D[D.mktDay == 1].activity_area_norm2020.mean()
                - D[D.mktDay == 0].activity_area_norm2020.mean() > 50))
        nd = D.groupby("mktID").n_market_days.first().value_counts()
        ok.append(("trading-day split is 104 / 32 / 1",
                [int(nd.get(k, 0)) for k in (1, 2, 3)] == [104, 32, 1]))
        print("validation")
        for name, passed in ok:
            print(f"   {'OK  ' if passed else 'FAIL'}  {name}")
        assert all(p for _, p in ok), "validation failed"


    market_images = build_market_images()

    _out = os.path.join(PROCESSED, "market_images.csv")
    market_images.to_csv(_out, index=False)
    df_mkt = pd.read_csv(os.path.join(PROCESSED, "market_images.csv"))
    df_mkt = df_mkt[~df_mkt['year'].isin([2017, 2026])].copy(deep=True)
    
    print(f"{len(market_images):,} images, {market_images.mktID.nunique()} markets, "
        f"{market_images.date.min().date()} to {market_images.date.max().date()}")
    print(f"  market-day images     {int((market_images.mktDay == 1).sum()):,}")
    print(f"  non-market-day images {int((market_images.mktDay == 0).sum()):,}")
    _nd = market_images.groupby("mktID").n_market_days.first().value_counts().sort_index()
    print("  trading days per week: " + ", ".join(f"{k} day(s): {v}" for k, v in _nd.items()))
    validate_market_images(market_images)
    print(f"\nwrote {_out}")
else :
    df_mkt = pd.read_csv(os.path.join(PROCESSED, "market_images.csv"))
    df_mkt = df_mkt[~df_mkt['year'].isin([2017, 2026])].copy(deep=True)

count    410784.000000
mean         17.634575
std          86.048224
min        -299.920980
25%         -33.176789
50%          10.951967
75%          69.746471
max         299.978040
Name: activity_area_norm2020, dtype: float64

count    410784.000000
mean         17.572395
std          85.914916
min        -299.920980
25%         -33.087805
50%          10.915673
75%          69.576333
max         299.978040
Name: activity_area_norm2020, dtype: float64

426,300 images, 137 markets, 2017-06-02 to 2026-02-02
  market-day images     79,342
  non-market-day images 346,958
  trading days per week: 1 day(s): 104, 2 day(s): 32, 3 day(s): 1
validation
   OK    137 markets
   OK    every market has a trading day
   OK    every market-day image falls on a declared trading day
   OK    one area per market
   OK    one coordinate pair per market
   OK    market days score far above non-market days
   OK    trading-day split is 104 / 32 / 1

wrote C:\Users\wb434633\WBG\Christina Wieser - Afghanistan\Geospatial information\MarketActivityIndex\processed_data\market_images.csv


## 2. General stats

In [ ]:
# ==== PAPER FIGURE 3, panel a: spatial distribution of periodic markets ====
# Provinces shaded by UN region, the 137 detected markets as dots.
# Output: paper_exhibits/fig03a_market_map.png

mk = df_mkt.drop_duplicates('mktID')
a = gpd.read_file(ADM1)
CSO = {'Central': 'Central', 'East': 'East', 'Northeast': 'NorthEast', 'North': 'NorthWest',
       'South': 'SouthEast', 'Southwest': 'SouthWest', 'West': 'West', 'WestCentral': 'West'}
a['region'] = a.CSO_Region.map(CSO)
d = mk[['mktID', 'marketlon', 'marketlat', 'un_region']]
S.save_data(d, 'fig03a_market_map')
S.save(P.fig03a_market_map(d, prov=a), 'fig03a_market_map')


In [ ]:
# ==== PAPER FIGURE 3, panel b: day of market activity ====
# Share of market days falling on each weekday (Sat-Fri), once-a-week markets vs
# markets trading more than once a week.
# Output: paper_exhibits/fig03b_days_of_operation.png

import re

pa = df_mkt.drop_duplicates('mktID')
days = pa.marketDays.apply(lambda x: {int(v) for v in re.findall(r'\d+', str(x))})

once = days[days.apply(len) == 1]; more = days[days.apply(len) > 1]
LAB = {6: 'Sat', 0: 'Sun', 1: 'Mon', 2: 'Tue', 3: 'Wed', 4: 'Thu', 5: 'Fri'}
ORD = [6, 0, 1, 2, 3, 4, 5]
c1 = pd.Series([d for s in once for d in s]).value_counts().reindex(ORD).fillna(0)
c2 = pd.Series([d for s in more for d in s]).value_counts().reindex(ORD).fillna(0)
s1, s2 = 100 * c1 / c1.sum(), 100 * c2 / c2.sum()
d = pd.concat([pd.DataFrame({'day': [LAB[k] for k in ORD], 'group': g, 'share': v.values, 'n': n})
               for g, v, n in [('Once-a-week markets', s1, len(once)),
                               ('More-than-once-a-week markets', s2, len(more))]])
S.save_data(d, 'fig03b_days_of_operation')
S.save(P.fig03b_days_of_operation(d), 'fig03b_days_of_operation')


## 3. MAI evolution 

In [22]:
TARGET     = 'activity_area_norm2020' 

def mai_series(d, label='mai', by=None):
    """The published nested-mean cascade: market-week, normalise each market to its
    own 2020 mean, then mean across markets within year-month, quarter, year."""
    # Keep only market days
    s = d.copy()
    s = s[s['mktDay']==1].copy()

    s['DataWithIssue'] = np.where(~s[TARGET].between(-5 , 300), 1, 0)
    s['DataWithIssue'] = np.where(s[TARGET].isnull(), np.nan, s['DataWithIssue'])

    s = s[s['DataWithIssue']==0].copy(deep=True)

    s[TARGET] = s.groupby(['date', 'mktID'])[TARGET].transform('mean')
    s.drop_duplicates(subset=['date', 'mktID'], inplace=True)

    s['aw'] = s.groupby(['mktID', 'year', 'month', 'week'])[TARGET].transform('mean')
    s = s.drop_duplicates(subset=['mktID', 'year', 'month', 'week'])
    avg = s[s.year == 2020].groupby('mktID').aw.mean()
    s['n'] = 100 * s.aw / s.mktID.map(avg)

    if by:                                    # optional intermediate grouping level
        s['n'] = s.groupby([by, 'year', 'month']).n.transform('mean')
        s = s.drop_duplicates(subset=[by, 'year', 'month'])
    s['n'] = s.groupby(['year', 'month']).n.transform('mean')
    s = s.drop_duplicates(subset=['year', 'month'])
    s['n'] = s.groupby(['year', 'quarter']).n.transform('mean')
    s = s.drop_duplicates(subset=['year', 'quarter'])
    return s.groupby('year').n.mean().rename(label)

mai_series(df_mkt)
# year
# 2018    102.089150
# 2019    100.448325
# 2020     99.996100
# 2021    108.990009
# 2022    110.384700
# 2023    103.167612
# 2024     95.433534
# 2025     88.472748


year
2018    102.089150
2019    100.468041
2020     99.996100
2021    108.990009
2022    110.384700
2023    103.167612
2024     95.433534
2025     88.472748
Name: mai, dtype: float64

In [ ]:
# ==== PAPER FIGURE 7, panel a: market activity and real GDP (2020 = 100) ====
# Output: paper_exhibits/fig07a_mai_gdp.png
g = pd.read_excel(MACRO, sheet_name='Real')
g['Sector'] = g['Sector'].ffill().astype(str).str.strip()
lvl = g[g['Measure'] == 'million Afs'].set_index('Sector')
cols = [2018, 2019, 2020, 2021, '2022p', 2023, 2024, 2025]
ser = lambda s: pd.Series(100 * lvl.loc[s, cols].astype(float).values
                          / lvl.loc[s, cols].astype(float).values[2], index=YEARS)
ag_gdp, tot_gdp = ser('Agriculture'), ser('Total')
mai = mai_series(df_mkt).reindex(YEARS)

d = pd.DataFrame({'year': YEARS, 'mai': mai.values, 'ag_gdp': ag_gdp.values,
                  'tot_gdp': tot_gdp.values})
S.save_data(d, 'fig07a_mai_gdp')
S.save(P.fig07a_mai_gdp(d), 'fig07a_mai_gdp')


In [ ]:
# ==== PAPER FIGURE 7, panel b: market activity and nighttime-light emissions ====
# National civilian nightlights (Feb-Jun), nightlights in the 10 km buffers of the
# periodic markets, and the MAI, all 2020 = 100 on one axis.
# Output: paper_exhibits/fig07b_mai_ntl.png
def ntl_national(months=range(2, 7)):
    """National civilian nightlights by year, summed over the given months (range
    excludes its upper bound: 2-6 = February to June).
    Reproduces exhibits/reference/04_ntl_national.csv."""
    n = pd.read_excel(NTL_ADU)
    n = n[n.month.isin(list(months)) & n.year.isin(YEARS)]
    return n.groupby('year').civilian_coalition.sum().rename('civilian_feb_jun')


def buffer_ntl_periodic(months=range(1, 13)):
    """National 10 km buffer nighttime lights (BNR-filtered), periodic markets
    only: sum across markets of each market's mean monthly buffer total over
    `months`. The source is ntl_bnr_periodic137_8years.csv.
    """
    n = pd.read_csv(os.path.join(PROCESSED, "ntl_bnr_periodic137_8years.csv"), low_memory=False)
    grp = pd.read_csv(os.path.join(PROCESSED, "market_groups.csv"), low_memory=False)
    n = n.merge(grp[["mkt_uid", "group"]], on="mkt_uid", how="left")
    n = n[n.group == "Periodic"]
    yrs = sorted({int(c[-6:-2]) for c in n.columns if c.startswith("SUM_BNR_")})
    out = {}
    for y in yrs:
        cols = [f"SUM_BNR_{y}{m:02d}" for m in months if f"SUM_BNR_{y}{m:02d}" in n.columns]
        out[y] = n[cols].mean(axis=1).sum()
    return pd.Series(out).sort_index()


mai = mai_series(df_mkt).reindex(YEARS)
ntl_nat = ntl_national()
ntl_idx = 100 * ntl_nat / ntl_nat.loc[2020]
buf = buffer_ntl_periodic()
buf_idx = 100 * buf / buf.loc[2020]

d = pd.DataFrame({'year': YEARS, 'mai': mai.values,
                  'ntl_total': ntl_idx.reindex(YEARS).values,
                  'ntl_buffer': buf_idx.reindex(YEARS).values})
S.save_data(d, 'fig07b_mai_ntl')
S.save(P.fig07b_mai_ntl(d), 'fig07b_mai_ntl')
